In [74]:
%run aaa_setup.ipynb

In [75]:
class Processor(SetUp): # type: ignore

    def __init__(self):
        super().__init__()
        return
    
    def filter(self, cutoff_institutions=None, cutoff_authors=None):
        self._filter_institutions(cutoff=cutoff_institutions)
        self._filter_authors(cutoff=cutoff_authors)
        self._modify_authorships()
        self._modify_works()
        self._modify_cites()
        return self
    
    def _filter_institutions(self, cutoff=None):
        sql = f"""
            CREATE OR REPLACE TABLE memory.institutions AS        
                WITH get_work_institution_CTE AS (
                    SELECT DISTINCT work_id, institution_id 
                    FROM project.authorships
                ),
                institution_counts AS (
                    SELECT
                        institution_id,
                        COUNT(work_id) AS works_count
                    FROM get_work_institution_CTE
                    GROUP BY institution_id
                ),
                ranked_institutions AS (
                    SELECT
                        ROW_NUMBER() OVER (ORDER BY works_count DESC) AS row_number,
                        institution_id,
                        works_count
                    FROM institution_counts
                )

                SELECT *
                    FROM ranked_institutions
                    WHERE works_count >= {cutoff}
                    ORDER BY works_count DESC
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_institutions FROM memory.institutions").show()
        self.db.sql("SELECT count(DISTINCT institution_id) AS count_institutions_original FROM project.authorships").show()
        return self

    def _filter_authors(self, cutoff=None):
        sql = f"""
            CREATE OR REPLACE TABLE memory.authors AS
                WITH get_work_author_CTE AS (
                    SELECT DISTINCT work_id, author_id, list_sort(list(publication_year))[1] AS first_year
                    FROM project.authorships
                    LEFT JOIN project.works
                    USING (work_id)
                    GROUP BY work_id, author_id
                ),
                author_counts_CTE AS (
                    SELECT
                        author_id,
                        COUNT(work_id)/(2026-first_year) AS works_count_prorata
                    FROM get_work_author_CTE
                    GROUP BY author_id, first_year
                ),
                ranked_authors_CTE AS (
                    SELECT
                        ROW_NUMBER() OVER (ORDER BY works_count_prorata DESC) AS row_number,
                        author_id,
                        works_count_prorata
                    FROM author_counts_CTE
                )

                SELECT *
                FROM ranked_authors_CTE
                WHERE works_count_prorata >= {cutoff}
                ORDER BY works_count_prorata DESC
        """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_authors FROM memory.authors").show()
        self.db.sql("SELECT count(DISTINCT author_id) AS count_authors_original FROM project.authorships").show()
        return self
    
    def _modify_authorships(self):
        sql = """
            CREATE OR REPLACE TABLE memory.authorships AS
            WITH
            filtered_works_authors_CTE AS
                (SELECT a.* 
                    FROM project.authorships a
                    LEFT JOIN memory.authors ma
                    ON a.author_id = ma.author_id)

            SELECT DISTINCT * 
                FROM filtered_works_authors_CTE a
                LEFT JOIN memory.institutions mi
                ON a.institution_id = mi.institution_id
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_authorships FROM memory.authorships").show()
        self.db.sql("SELECT count(DISTINCT work_id) AS count_authorships_original FROM project.authorships").show()      

    def _modify_works(self):
        sql = """
            CREATE OR REPLACE TABLE memory.works AS
            SELECT w.* 
                FROM project.works w
                INNER JOIN memory.authorships
                USING (work_id)
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_works FROM memory.works").show()
        self.db.sql("SELECT count(DISTINCT work_id) AS count_works_original FROM project.works").show()
        return self

  

    def _modify_cites(self):
        ...  

In [76]:
def main():

    p = Processor()
    p.filter(cutoff_institutions=150, cutoff_authors=2)

    return

In [77]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────────┬───────────┐
│ database │ schema  │         name         │     column_names     │             column_types              │ temporary │
│ varchar  │ varchar │       varchar        │      varchar[]       │               varchar[]               │  boolean  │
├──────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────────┼───────────┤
│ memory   │ main    │ t1                   │ [i, j]               │ [INTEGER, INTEGER]                    │ false     │
│ project  │ main    │ author_works_counts  │ [author_id, author…  │ [VARCHAR, VARCHAR, BIGINT, BIGINT, …  │ false     │
│ project  │ main    │ authors              │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ project  │ main    │ authorships          │ [work_id, author_i…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ project  │ main    │ citation_